In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
SEED = 42
processed_dir = Path('../data/processed')
manifest_path = processed_dir / 'manifest.csv'



In [6]:
manifest =pd.read_csv(manifest_path)
print('total tiles', len(manifest))
display(manifest.head())


total tiles 2094


,tile_id,lidar_path,building_mask,platform_mask,aguada_mask,building_pixels,platform_pixels,aguada_pixels,has_building,has_platform,has_aguada,center_x,center_y
0,0,../data/raw/lidar/tile_0_lidar.tif,../data/raw/masks/tile_0_mask_building.tif,../data/raw/masks/tile_0_mask_platform.tif,../data/raw/masks/tile_0_mask_aguada.tif,0,0,0,False,False,False,240.0,240.0
1,1,../data/raw/lidar/tile_1_lidar.tif,../data/raw/masks/tile_1_mask_building.tif,../data/raw/masks/tile_1_mask_platform.tif,../data/raw/masks/tile_1_mask_aguada.tif,7620,47877,0,True,True,False,240.0,240.0
2,2,../data/raw/lidar/tile_2_lidar.tif,../data/raw/masks/tile_2_mask_building.tif,../data/raw/masks/tile_2_mask_platform.tif,../data/raw/masks/tile_2_mask_aguada.tif,14778,24129,0,True,True,False,240.0,240.0
3,3,../data/raw/lidar/tile_3_lidar.tif,../data/raw/masks/tile_3_mask_building.tif,../data/raw/masks/tile_3_mask_platform.tif,../data/raw/masks/tile_3_mask_aguada.tif,0,0,0,False,False,False,240.0,240.0
4,4,../data/raw/lidar/tile_4_lidar.tif,../data/raw/masks/tile_4_mask_building.tif,../data/raw/masks/tile_4_mask_platform.tif,../data/raw/masks/tile_4_mask_aguada.tif,0,0,0,False,False,False,240.0,240.0


In [7]:
class_cols = ['has_building', 'has_platform', 'has_aguada']
for col in class_cols:
    count = manifest[col].sum()
    print(f'{col}: {count} ({count/len(manifest):.2%})')

has_building: 1343 (64.14%)
has_platform: 1131 (54.01%)
has_aguada: 78 (3.72%)


## Create a row in manifest to determine splitting

In [ ]:
def make_strat(row):
    b = int(row['has_building'])
    p = int(row['has_platform'])
    a = int(row['has_aguada'])
    return f'{b}{p}{a}'

manifest['stratum'] = manifest.apply(make_strat, axis=1)


In [10]:
strat_counts = manifest['stratum'].value_counts().sort_index()
print(strat_counts)

stratum
000     652
001      35
010      62
011       2
100     269
101       7
110    1033
111      34
Name: count, dtype: int64


## Create splitted data without leakage

In [11]:
X = manifest.index.to_numpy()
y = manifest[['has_building', 'has_platform', 'has_aguada']].astype(int).to_numpy()

In [14]:
split1 = MultilabelStratifiedShuffleSplit(n_splits = 1, test_size = .3, random_state = SEED)
train_idx, test_idx = next(split1.split(X, y))
train_df = manifest.iloc[train_idx].copy()
temp_df = manifest.iloc[test_idx].copy()
X_temp = temp_df.index.to_numpy()
y_temp = temp_df[['has_building', 'has_platform', 'has_aguada']].astype(int).to_numpy()

In [15]:
split2 = MultilabelStratifiedShuffleSplit(n_splits = 1, test_size = .5, random_state = SEED)
val_idx, test_idx = next(split2.split(X_temp, y_temp))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

In [16]:
def show_distribution(name, df):
    print(f'{name}: {len(df)}')
    for col in class_cols:
        count = df[col].sum()
        percent = count / len(df) * 100
        print(f'  {col}: {count} ({percent:.2f}%)')
show_distribution('Full Dataset', manifest)
show_distribution('Train Set', train_df)
show_distribution('Validation Set', val_df)
show_distribution('Test Set', test_df)

Full Dataset: 2094
  has_building: 1343 (64.14%)
  has_platform: 1131 (54.01%)
  has_aguada: 78 (3.72%)
Train Set: 1465
  has_building: 940 (64.16%)
  has_platform: 791 (53.99%)
  has_aguada: 55 (3.75%)
Validation Set: 314
  has_building: 201 (64.01%)
  has_platform: 170 (54.14%)
  has_aguada: 11 (3.50%)
Test Set: 315
  has_building: 202 (64.13%)
  has_platform: 170 (53.97%)
  has_aguada: 12 (3.81%)


In [17]:
print("TRAIN")
print(train_df["stratum"].value_counts().sort_index())

print("\nVALIDATION")
print(val_df["stratum"].value_counts().sort_index())

print("\nTEST")
print(test_df["stratum"].value_counts().sort_index())

TRAIN
stratum
000    460
001     22
010     42
011      1
100    187
101      5
110    721
111     27
Name: count, dtype: int64

VALIDATION
stratum
000     94
001      8
010     11
100     42
110    156
111      3
Name: count, dtype: int64

TEST
stratum
000     98
001      5
010      9
011      1
100     40
101      2
110    156
111      4
Name: count, dtype: int64


In [ ]:
manifest['split'] = ''
manifest.loc[manifest['tile_id'].isin(train_df['tile_id']), 'split'] = 'train'
manifest.loc[manifest['tile_id'].isin(val_df['tile_id']), 'split'] = 'val'
manifest.loc[manifest['tile_id'].isin(test_df['tile_id']), 'split'] = 'test'
manifest.to_csv(processed_dir / 'manifest_split.csv', index=False)
print(manifest['split'].value_counts())
print(manifest['split'].isna().sum())


split
train    1465
test      315
val       314
Name: count, dtype: int64
0
       has_building  has_platform  has_aguada
split                                        
test       0.641270      0.539683    0.038095
train      0.641638      0.539932    0.037543
val        0.640127      0.541401    0.035032
